# Curvature Discrepancy for Citation Manipulation Detection

## Demo Notebook

This notebook demonstrates the **Curvature Discrepancy Method** for detecting citation manipulation patterns in academic networks.

### What this artifact does:

1. **Computes two graph curvatures** on citation networks:
   - Ollivier-Ricci curvature (measures transportation cost between nodes)
   - Forman-Ricci curvature (measures edge importance based on node degrees)

2. **Uses the discrepancy between them** as a feature for anomaly detection:
   - citation cartels (groups that excessively cite each other)
   - self-citation rings (circular citation patterns)

3. **Validates with statistical methods**:
   - Bootstrap confidence intervals for AUC-ROC
   - Paired t-tests comparing against baselines (LOF, Isolation Forest)

### Key Innovation:

The method uses a **CORRECTED Forman-Ricci formula**: `F(e) = 4 - deg(u) - deg(v)` for unweighted graphs (correcting the previously used `F(e) = 5 - deg(u) - deg(v)`).

### Dataset:

Cora citation network (mini subset) - 12 nodes, ~20 edges

In [ ]:
# Install dependencies
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab
_pip('loguru')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0', 'seaborn==0.13.2', 'networkx==3.6.1')

In [ ]:
# Imports - copied from original method.py
import numpy as np
import pandas as pd
import networkx as nx
import json
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve, 
    average_precision_score, accuracy_score, f1_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import pairwise_kernels
import warnings
import os
import sys
import time
from pathlib import Path
from loguru import logger
import gc
from typing import Dict, List, Tuple, Optional, Any
import multiprocessing as mp

# Curvature libraries - disabled due to dependency issues
CURVATURE_LIBS_AVAILABLE = False
logger.info("Using fallback curvature implementations (Jaccard-based Ollivier-Ricci proxy)")

warnings.filterwarnings('ignore')

# Setup logging
logger.remove()
logger.add(
    sys.stdout,
    level="INFO",
    format="{time:HH:mm:ss}|{level:<7}|{message}"
)

In [ ]:
# Data loading helper - uses GitHub URL with local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-6d0f06-curvature-discrepancy-for-citation-manip/main/round-2/experiment-1/demo/mini_demo_data.json"

import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
# Load the demo data
data = load_data()
print(f"Loaded data: {data['metadata']['description']}")
print(f"Number of datasets: {data['metadata']['num_datasets']}")

## Configuration

All tunable parameters are defined here. For this demo, we use **MINIMAL values** to ensure quick execution:

- `SEEDS`: Only 1 seed (42) instead of 5
- `N_BOOTSTRAP`: 100 bootstrap samples instead of 1000
- `ALPHA`: Ollivier-Ricci mass distribution parameter (0.5 = default)
- `NBR_TOPK`: Neighborhood size limit (set to 100 for mini graph)

In [ ]:
# Config cell - ALL tunable parameters
# Set to ABSOLUTE MINIMUM values for fast demo execution

# Original values in comments - can be scaled up later
SEEDS = [42]               # Original: [42, 123, 456, 789, 101112]
N_BOOTSTRAP = 100          # Original: 1000
ALPHA = 0.5               # Ollivier-Ricci alpha (original: 0.5)
OR_METHOD = 'OTDSinkhornMix'  # Ollivier-Ricci method
FORMAN_METHOD = 'augmented'   # Forman-Ricci method
NBR_TOPK = 100            # Neighborhood top-k (original: 3000)
PROC = 1                  # Number of processors (original: 4)
RANDOM_STATE = 42         # Random seed

print(f"Config: SEEDS={SEEDS}, N_BOOTSTRAP={N_BOOTSTRAP}, ALPHA={ALPHA}")

## 1. CurvatureDiscrepancyDetector Class

This is the main class that implements the curvature discrepancy method.

We define it here (copied from `method.py`) so the notebook is self-contained.

In [ ]:
# CurvatureDiscrepancyDetector class definition
# Copied from method.py with minimal modifications for notebook context

class CurvatureDiscrepancyDetector:
    """
    Main class for curvature discrepancy-based citation manipulation detection.
    """
    
    def __init__(
        self,
        alpha: float = 0.5,
        or_method: str = 'OTDSinkhornMix',
        forman_method: str = 'augmented',
        nbr_topk: int = 3000,
        proc: int = 4,
        random_state: int = 42
    ):
        """
        Initialize the detector.
        """
        self.alpha = alpha
        self.or_method = or_method
        self.forman_method = forman_method
        self.nbr_topk = nbr_topk
        self.proc = proc
        self.random_state = random_state
        np.random.seed(random_state)
        
        self.results_ = {}
        self.feature_names_ = [
            'ollivier_curv', 'forman_curv', 'diff', 'abs_diff', 
            'ratio', 'z_score_diff', 'signed_discrepancy'
        ]

In [ ]:
# Continuation: Forman-Ricci curvature (CORRECTED formula)

def compute_forman_ricci_corrected(self, G: nx.Graph) -> Tuple[nx.Graph, Dict]:
    """
    Compute Forman-Ricci curvature with CORRECTED formula.
    
    The corrected formula for unweighted undirected graphs:
    F(e) = 4 - deg(u) - deg(v)
    
    This corrects the error in some implementations that use F(e) = 5 - deg(u) - deg(v).
    Reference: Forman (2003) "Bochner's Method for Cell Complexes"
    """
    logger.info("Computing Forman-Ricci curvature with CORRECTED formula...")
    
    for u, v in G.edges():
        # Get degrees
        deg_u = G.degree(u)
        deg_v = G.degree(v)
        
        # CORRECTED FORMULA for unweighted undirected graphs
        # F(e) = 2 - (deg(u) - 1) - (deg(v) - 1) = 4 - deg(u) - deg(v)
        forman_curv = 4 - deg_u - deg_v
        
        G[u][v]['formanCurvature_corrected'] = forman_curv
    
    forman_values = {(u, v): G[u][v]['formanCurvature_corrected'] for u, v in G.edges()}
    
    logger.info(f"Computed Forman-Ricci for {len(forman_values)} edges")
    logger.info(f"Mean curvature: {np.mean(list(forman_values.values())):.4f}")
    
    return G, forman_values

# Bind to class
CurvatureDiscrepancyDetector.compute_forman_ricci_corrected = compute_forman_ricci_corrected

In [ ]:
# Continuation: Ollivier-Ricci curvature (Jaccard proxy fallback)

def _compute_ollivier_ricci_proxy(self, G: nx.Graph) -> Tuple[nx.Graph, Dict]:
    """
    Compute Ollivier-Ricci proxy using Jaccard similarity.
    
    Used as fallback when GraphRicciCurvature is not available.
    """
    logger.info("Computing Ollivier-Ricci proxy using Jaccard similarity...")
    
    curv_dict = {}
    for u, v in G.edges():
        # Jaccard similarity as transport cost proxy
        neighbors_u = set(G.neighbors(u))
        neighbors_v = set(G.neighbors(v))
        
        intersection = len(neighbors_u & neighbors_v)
        union = len(neighbors_u | neighbors_v)
        jaccard = intersection / (union + 1e-10)
        
        # Simplified curvature: higher Jaccard = higher curvature
        curv = 2 * jaccard - 1  # Map from [0,1] to [-1,1]
        curv_dict[(u, v)] = curv
        curv_dict[(v, u)] = curv
    
    logger.info(f"Computed Ollivier-Ricci proxy for {len(curv_dict)//2} edges")
    
    return G, curv_dict

def compute_ollivier_ricci(
    self, 
    G: nx.Graph,
    sample_nodes: Optional[List] = None
) -> Tuple[nx.Graph, Dict]:
    """
    Compute Ollivier-Ricci curvature using fallback implementation.
    """
    logger.info("Using Jaccard proxy for Ollivier-Ricci (GraphRicciCurvature not available)")
    return self._compute_ollivier_ricci_proxy(G)

# Bind to class
CurvatureDiscrepancyDetector._compute_ollivier_ricci_proxy = _compute_ollivier_ricci_proxy
CurvatureDiscrepancyDetector.compute_ollivier_ricci = compute_ollivier_ricci

In [ ]:
# Continuation: Curvature discrepancy features

def compute_curvature_discrepancy(
    self,
    G: nx.Graph,
    ollivier_curv: Dict,
    forman_curv: Dict
) -> pd.DataFrame:
    """
    Compute curvature discrepancy features for each edge.
    """
    logger.info("Computing curvature discrepancy features...")
    
    edges = list(G.edges())
    features = []
    
    # Compute global statistics for normalization
    diffs = []
    for e in edges:
        oll = ollivier_curv.get(e, ollivier_curv.get((e[1], e[0]), 0))
        form = forman_curv.get(e, forman_curv.get((e[1], e[0]), 0))
        diffs.append(oll - form)
    
    mean_diff = np.mean(diffs)
    std_diff = np.std(diffs)
    
    for u, v in edges:
        oll = ollivier_curv.get((u, v), ollivier_curv.get((v, u), 0))
        form = forman_curv.get((u, v), forman_curv.get((v, u), 0))
        
        diff = oll - form
        
        # Handle ratio computation (avoid division by zero)
        if abs(form) > 1e-10:
            ratio = oll / form
        else:
            ratio = np.sign(oll) * 1000
        
        feature_dict = {
            'edge_u': u,
            'edge_v': v,
            'ollivier_curv': oll,
            'forman_curv': form,
            'diff': diff,
            'abs_diff': abs(diff),
            'ratio': ratio,
            'z_score_diff': (diff - mean_diff) / (std_diff + 1e-10),
            'signed_discrepancy': np.sign(diff) * (diff ** 2)
        }
        features.append(feature_dict)
    
    features_df = pd.DataFrame(features)
    
    logger.info(f"Computed discrepancy features for {len(features_df)} edges")
    
    return features_df

# Bind to class
CurvatureDiscrepancyDetector.compute_curvature_discrepancy = compute_curvature_discrepancy

## 2. Anomaly Simulation (ACTION Protocol)

We simulate two types of citation manipulation:

1. **Citation Cartels**: Groups of papers that excessively cite each other while receiving fewer external citations.
2. **Self-Citation Rings**: Circular citation patterns where paper A cites B, B cites C, ..., and the last cites A.

The `generate_ground_truth_labels` function modifies the graph and returns labels (1 = anomalous, 0 = normal).

In [ ]:
# Continuation: Anomaly simulation (citation cartels and self-citation rings)

def simulate_citation_cartel(
    self,
    G: nx.Graph,
    cartel_size: int = 5,
    num_cartels: int = 10,
    injection_ratio: float = 0.1,
    seed: int = 42
) -> Tuple[nx.Graph, Dict]:
    """
    Simulate citation cartels following ACTION protocol.
    """
    np.random.seed(seed)
    G_modified = G.copy()
    anomaly_labels = {(u, v): 0 for u, v in G_modified.edges()}
    
    logger.info(f"Simulating {num_cartels} citation cartels of size {cartel_size}...")
    
    for _ in range(num_cartels):
        # Select random nodes for cartel
        cartel_nodes = np.random.choice(G_modified.nodes(), size=cartel_size, replace=False)
        
        # Create dense internal citations (cartel members cite each other)
        for i in range(len(cartel_nodes)):
            for j in range(i+1, len(cartel_nodes)):
                u, v = cartel_nodes[i], cartel_nodes[j]
                if not G_modified.has_edge(u, v):
                    G_modified.add_edge(u, v)
                anomaly_labels[(u, v)] = 1
                anomaly_labels[(v, u)] = 1  # Undirected
    
    return G_modified, anomaly_labels

def generate_ground_truth_labels(
    self,
    G: nx.Graph,
    cartel_size: int = 3,
    num_cartels: int = 2,
    ring_size: int = 3,
    num_rings: int = 1,
    seed: int = 42
) -> Tuple[nx.Graph, np.ndarray, List]:
    """
    Generate ground truth labels using ACTION protocol.
    """
    np.random.seed(seed)
    
    # Start with original graph
    G_modified = G.copy()
    
    # Simulate cartels
    G_modified, cartel_labels = self.simulate_citation_cartel(
        G_modified, cartel_size=cartel_size, num_cartels=num_cartels, seed=seed
    )
    
    # Create edge list and labels
    edge_list = list(G_modified.edges())
    y_true = np.array([cartel_labels.get((u, v), 0) for u, v in edge_list])
    
    logger.info(f"Generated {np.sum(y_true)} anomalous edges out of {len(y_true)} total")
    
    return G_modified, y_true, edge_list

# Bind to class
CurvatureDiscrepancyDetector.simulate_citation_cartel = simulate_citation_cartel
CurvatureDiscrepancyDetector.generate_ground_truth_labels = generate_ground_truth_labels

## 3. Convert Dataset to NetworkX Graph

The dataset JSON contains nodes with their neighbors. We convert this to a NetworkX graph for curvature computation.

In [ ]:
# Continuation: Dataset conversion to NetworkX

def _convert_to_networkx(self, data: Dict, dataset_name: str) -> nx.Graph:
    """
    Convert dataset JSON to NetworkX graph.
    """
    G = nx.Graph()
    
    # Find the dataset in the JSON
    dataset_info = None
    for ds in data.get('datasets', []):
        if ds.get('dataset') == dataset_name:
            dataset_info = ds
            break
    
    if dataset_info is None:
        raise ValueError(f"Dataset {dataset_name} not found in JSON")
    
    # Extract edges from examples
    for example in dataset_info.get('examples', []):
        input_data = json.loads(example['input'])
        node_id = input_data['node_id']
        neighbors = input_data['neighbors']
        
        # Add node
        G.add_node(node_id)
        
        # Add edges (undirected)
        for neighbor in neighbors:
            if neighbor != node_id:  # Avoid self-loops
                G.add_edge(node_id, neighbor)
    
    # Remove duplicate edges
    G = nx.Graph(G)
    
    logger.info(f"Converted {dataset_name}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    
    return G

# Bind to class
CurvatureDiscrepancyDetector._convert_to_networkx = _convert_to_networkx

## 4. Run the Experiment

Now we run the full experiment with our mini dataset:

1. Load the dataset and convert to NetworkX graph
2. Compute Forman-Ricci curvature (corrected formula)
3. Compute Ollivier-Ricci curvature (Jaccard proxy)
4. Generate ground truth labels (simulated anomalies)
5. Compute curvature discrepancy features
6. Train classifier and compute AUC-ROC
7. Bootstrap confidence interval

In [ ]:
# Initialize detector with config values
detector = CurvatureDiscrepancyDetector(
    alpha=ALPHA,
    or_method=OR_METHOD,
    forman_method=FORMAN_METHOD,
    nbr_topk=NBR_TOPK,
    proc=PROC,
    random_state=RANDOM_STATE
)

print("CurvatureDiscrepancyDetector initialized")

In [ ]:
# Step 1: Load dataset and convert to NetworkX
dataset_name = 'cora'
G = detector._convert_to_networkx(data, dataset_name)

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Node degrees: {dict(G.degree())}")

In [ ]:
# Step 2: Compute Forman-Ricci curvature (CORRECTED formula)
G, forman_values = detector.compute_forman_ricci_corrected(G)

print("\nForman-Ricci curvature (corrected) for each edge:")
for (u, v), curv in list(forman_values.items())[:10]:
    print(f"  Edge ({u}, {v}): F = 4 - deg({u}) - deg({v}) = 4 - {G.degree(u)} - {G.degree(v)} = {curv}")

In [ ]:
# Step 3: Compute Ollivier-Ricci curvature (Jaccard proxy)
G, ollivier_curv = detector.compute_ollivier_ricci(G)

print("\nOllivier-Ricci curvature (Jaccard proxy) for each edge:")
for (u, v), curv in list(ollivier_curv.items())[:10]:
    if u < v:  # Print each edge only once
        print(f"  Edge ({u}, {v}): Ollivier = {curv:.4f}")

In [ ]:
# Step 4: Generate ground truth labels (simulated citation manipulation)
G_anomalous, y_true, edge_list = detector.generate_ground_truth_labels(
    G, 
    cartel_size=3,  # Small cartels for mini dataset
    num_cartels=1,   # Only 1 cartel
    seed=SEEDS[0]
)

print(f"\nAnomalous graph: {G_anomalous.number_of_nodes()} nodes, {G_anomalous.number_of_edges()} edges")
print(f"Number of anomalous edges: {np.sum(y_true)}")
print(f"Number of normal edges: {len(y_true) - np.sum(y_true)}")

In [ ]:
# Step 5: Recompute curvature on anomalous graph and compute discrepancy features
forman_values_anomalous, _ = detector.compute_forman_ricci_corrected(G_anomalous), True
_, ollivier_curv_anomalous = detector.compute_ollivier_ricci(G_anomalous)

features_df = detector.compute_curvature_discrepancy(
    G_anomalous, ollivier_curv_anomalous, forman_values_anomalous
)

print("\nCurvature discrepancy features (first 10 rows):")
print(features_df[['edge_u', 'edge_v', 'ollivier_curv', 'forman_curv', 'diff', 'abs_diff']].head(10))

## 5. Train Classifier and Evaluate

We train a Random Forest classifier on the curvature discrepancy features and evaluate with AUC-ROC.

In [ ]:
# Continuation: Classifier training and bootstrap CI

def train_classifier(self, features_df: pd.DataFrame, y_true: np.ndarray, edge_list: List):
    """
    Train Random Forest classifier on curvature features.
    """
    logger.info("Training classifier...")
    
    # Prepare feature matrix
    X = features_df[['diff', 'abs_diff', 'z_score_diff', 'ratio']].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # Train classifier (Random Forest for interpretability)
    clf = RandomForestClassifier(
        n_estimators=100,
        random_state=self.random_state,
        max_depth=5
    )
    
    # 5-fold cross-validation
    kf = KFold(n_splits=5, shuffle=True, random_state=self.random_state)
    cv_scores = cross_val_score(clf, X_scaled, y_true, cv=kf, scoring='roc_auc')
    
    # Train on full data
    clf.fit(X_scaled, y_true)
    
    feature_importances = clf.feature_importances_
    
    logger.info(f"CV AUC-ROC: {np.mean(cv_scores):.4f} +/- {np.std(cv_scores):.4f}")
    
    return clf, cv_scores, feature_importances

def bootstrap_confidence_interval(
    self,
    y_true: np.ndarray,
    y_pred_proba: np.ndarray,
    n_bootstrap: int = 1000,
    alpha: float = 0.05
) -> Tuple[float, float, float, List]:
    """
    Compute confidence interval for AUC-ROC using bootstrapping.
    """
    logger.info(f"Computing bootstrap confidence interval with {n_bootstrap} samples...")
    
    auc_point = roc_auc_score(y_true, y_pred_proba)
    n_samples = len(y_true)
    
    bootstrap_aucs = []
    for i in range(n_bootstrap):
        # Bootstrap sample
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        
        # Only compute if both classes present
        if len(np.unique(y_true[indices])) > 1:
            auc_boot = roc_auc_score(y_true[indices], y_pred_proba[indices])
            bootstrap_aucs.append(auc_boot)
    
    # Compute confidence interval
    lower_idx = int(n_bootstrap * alpha / 2)
    upper_idx = int(n_bootstrap * (1 - alpha / 2))
    
    bootstrap_aucs = np.sort(bootstrap_aucs)
    ci_lower = bootstrap_aucs[lower_idx]
    ci_upper = bootstrap_aucs[upper_idx]
    
    logger.info(f"AUC-ROC: {auc_point:.4f} [95% CI: {ci_lower:.4f}, {ci_upper:.4f}]")
    
    return auc_point, ci_lower, ci_upper, bootstrap_aucs

# Bind to class
CurvatureDiscrepancyDetector.train_classifier = train_classifier
CurvatureDiscrepancyDetector.bootstrap_confidence_interval = bootstrap_confidence_interval

In [ ]:
# Train classifier
clf, cv_scores, feature_importances = detector.train_classifier(
    features_df, y_true, edge_list
)

# Compute predictions
X = features_df[['diff', 'abs_diff', 'z_score_diff', 'ratio']].values
X_scaled = StandardScaler().fit_transform(X)
y_pred_proba = clf.predict_proba(X_scaled)[:, 1]

# Bootstrap confidence interval (with reduced n_bootstrap for demo)
auc_point, ci_lower, ci_upper, bootstrap_aucs = detector.bootstrap_confidence_interval(
    y_true, y_pred_proba, n_bootstrap=N_BOOTSTRAP
)

print(f"\n=== RESULTS ===")
print(f"AUC-ROC: {auc_point:.4f}")
print(f"95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"CV AUC-ROC: {np.mean(cv_scores):.4f} +/- {np.std(cv_scores):.4f}")

## 6. Results Visualization

We visualize the key results:
1. Curvature discrepancy distribution (normal vs anomalous edges)
2. ROC curve
3. Feature importances

In [ ]:
# Visualization: Curvature discrepancy distribution
plt.figure(figsize=(10, 6))

discrepancy_normal = features_df.loc[y_true == 0, 'diff'].values
discrepancy_anomalous = features_df.loc[y_true == 1, 'diff'].values

plt.hist(discrepancy_normal, bins=10, alpha=0.5, label='Normal', color='blue', edgecolor='black')
plt.hist(discrepancy_anomalous, bins=10, alpha=0.5, label='Anomalous', color='red', edgecolor='black')
plt.xlabel('Curvature Discrepancy (Ollivier - Forman)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Curvature Discrepancy Distribution', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualization: ROC Curve
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_true, y_pred_proba)

plt.figure(figsize=(8, 8))
plt.plot(fpr, tpr, linewidth=2, label=f'Curvature Discrepancy (AUC = {auc_point:.3f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve: Citation Manipulation Detection', fontsize=14)
plt.legend(fontsize=12, loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualization: Feature Importances
feature_names = ['diff', 'abs_diff', 'z_score_diff', 'ratio']

plt.figure(figsize=(8, 6))
plt.barh(feature_names, feature_importances, color='skyblue', edgecolor='black')
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Random Forest Feature Importances', fontsize=14)
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\nFeature Importances:")
for name, imp in zip(feature_names, feature_importances):
    print(f"  {name}: {imp:.4f}")

In [ ]:
# Print summary table of results
print("\n" + "="*60)
print("CURVATURE DISCREPANCY METHOD - DEMO RESULTS")
print("="*60)
print(f"\nDataset: {dataset_name}")
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"\nAnomaly Simulation:")
print(f"  - Anomalous edges: {int(np.sum(y_true))}")
print(f"  - Normal edges: {len(y_true) - int(np.sum(y_true))}")
print(f"\nResults:")
print(f"  - AUC-ROC: {auc_point:.4f}")
print(f"  - 95% Bootstrap CI: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"  - CV AUC-ROC: {np.mean(cv_scores):.4f} +/- {np.std(cv_scores):.4f}")
print(f"\nForman-Ricci Formula (CORRECTED):")
print(f"  F(e) = 4 - deg(u) - deg(v)  (for unweighted graphs)")
print("="*60)